# ノートブック① Genesis 単体デモ

Genesis 物理シミュレータ（`genesis-world`）を Google Colab 上でヘッドレス実行し、
Franka Panda アームのシミュレーション動画（mp4）を生成するデモです。

## 前提
- **GPU ランタイム必須**: メニューの「ランタイム」→「ランタイムのタイプを変更」→ ハードウェアアクセラレータで **T4 GPU** を選択してください。
- 所要時間: 約 10 分（インストール込み）
- 外部アセット・API キーは不要です。

## 注意
- Colab では画面（ディスプレイ）が無いため、Genesis のインタラクティブビューアは使えません。
  本ノートブックはカメラのオフスクリーン描画で動画を生成します。

上から順にセルを実行してください。

In [ ]:
# === セル2: GPU 確認 ===
# GPU ランタイムでない場合はここで停止する（後続のインストールが無駄になるため）
!nvidia-smi

import torch                       # Colab プリインストールの PyTorch
assert torch.cuda.is_available(), (
    "CUDA GPU が見つかりません。メニューの「ランタイム」→「ランタイムのタイプを変更」で GPU を選択し、"
    "ランタイムを再起動してから最初のセルから実行し直してください。"
)
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# === セル3: 本リポジトリの clone ===
import os                          # パス操作用

REPO_URL = "https://github.com/ringo7pie/RoboGen-GenesisWorld.git"   # 本リポジトリの公開 URL
REPO_DIR = "/content/RoboGen-GenesisWorld"                            # clone 先ディレクトリ

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("既に clone 済みのためスキップします。")

%cd {REPO_DIR}

In [ ]:
# === セル4: Genesis のインストール ===
# 所要 2〜3 分。torch は Colab プリインストール版を流用する（再インストールしない）
!bash scripts/setup_genesis.sh

In [ ]:
# === セル5: セルフチェック ===
# GPU / import / バージョン を検査し、OK/NG の表を出力する。
# NG がある場合は、このセルの出力全体をエラー報告として共有してください。
!python scripts/check_env.py --stage genesis

In [ ]:
# === セル6: Genesis の初期化 ===
import genesis as gs               # Genesis 本体

try:
    gs.init(backend=gs.gpu)        # GPU (CUDA) バックエンドで初期化
    print("GPU バックエンドで初期化しました。")
except Exception as e:
    print(f"GPU 初期化に失敗したため CPU にフォールバックします: {e}")
    gs.init(backend=gs.cpu)        # CPU フォールバック（動作はするが低速）

In [ ]:
# === セル7: シーン構築（地面 + Franka アーム + 箱 + カメラ） ===
import numpy as np                 # 関節目標の計算用

scene = gs.Scene(
    show_viewer=False,             # Colab はディスプレイが無いためビューアは無効（必須）
    sim_options=gs.options.SimOptions(dt=0.01),   # シミュレーション刻み幅 10ms
)

plane = scene.add_entity(gs.morphs.Plane())       # 地面（無限平面）
franka = scene.add_entity(
    gs.morphs.MJCF(file="xml/franka_emika_panda/panda.xml"),   # Genesis 同梱の Franka Panda モデル
)
box = scene.add_entity(
    gs.morphs.Box(size=(0.06, 0.06, 0.06), pos=(0.6, 0.0, 0.03)),   # アーム前方に置く小箱
)

camera = scene.add_camera(
    res=(640, 480),                # 解像度
    pos=(2.5, -1.5, 1.5),          # カメラ位置
    lookat=(0.5, 0.0, 0.3),        # 注視点（アームと箱のあたり）
    fov=35,                        # 視野角（度）
    GUI=False,                     # オフスクリーン描画（ヘッドレス必須）
)

scene.build()                      # シーンの構築（これ以降エンティティ追加は不可）
print(f"シーン構築完了。Franka の自由度数: {franka.n_dofs}")

In [ ]:
# === セル8: シミュレーション + カメラ録画 ===
import sys                         # モジュール検索パスの追加用
sys.path.append(os.path.join(REPO_DIR, "scripts"))   # 共通ユーティリティを import 可能にする
from genesis_record import record_simulation          # 録画ユーティリティ（本リポジトリ製）

n_dofs = franka.n_dofs             # Franka の自由度数（アーム7 + グリッパ2 = 9）
home_pos = np.zeros(n_dofs)        # 基準姿勢（全関節 0）


# 毎ステップ呼ばれる制御関数: 各関節を位相をずらした正弦波で駆動する（デモ用の見栄え動作）
def wave_control(step_i):
    t = step_i * 0.01                                   # 経過時間（秒）
    target = home_pos.copy()                            # 目標関節角度
    for j in range(min(7, n_dofs)):                     # アームの 7 関節のみ駆動
        target[j] = 0.5 * np.sin(1.5 * t + j * 0.5)     # 振幅 0.5rad・位相ずらしの正弦波
    franka.control_dofs_position(target)                # 位置制御で目標を指令


video_path, n_frames = record_simulation(
    scene=scene,
    camera=camera,
    steps=600,                     # 600 ステップ = シミュレーション時間 6 秒
    out_path="outputs/genesis_demo.mp4",   # 出力する動画ファイル
    fps=30,                        # 動画のフレームレート
    control_fn=wave_control,       # 上で定義した制御関数
    render_every=2,                # 2 ステップごとに 1 フレーム録画
)

In [ ]:
# === セル9: 動画の表示 + 検証 ===
from IPython.display import Video   # ノートブック内での動画再生用

size_kb = os.path.getsize(video_path) / 1024   # 動画ファイルサイズ (KB)
assert size_kb > 10, f"動画が小さすぎます ({size_kb:.1f} KB)。録画に失敗している可能性があります。"
assert n_frames > 100, f"フレーム数が不足しています ({n_frames})。"
print(f"検証 OK: {video_path} ({size_kb:.1f} KB, {n_frames} フレーム)")

Video(video_path, embed=True, width=640)   # 動画をインライン表示

In [ ]:
# === セル10: （任意）Google Drive への保存 ===
# 動画を残したい場合のみ実行してください。Drive のマウント許可を求められます
# （許可画面では「すべて選択」にチェックして続行。一部のみの許可では mount failed になります）。
# このセルは単独実行に耐えるよう自己完結にしてある（ランタイム再起動後でも動く）。
from google.colab import drive     # Drive マウント用
import os                          # パス操作用
import shutil                      # ファイルコピー用

drive.mount("/content/drive")
video_path = "/content/RoboGen-GenesisWorld/outputs/genesis_demo.mp4"   # セル8 が出力した動画
assert os.path.isfile(video_path), (
    "動画ファイルがありません。ランタイム再起動で消えた場合はセル6〜8を再実行してください。"
)
save_dir = "/content/drive/MyDrive/robogen_outputs"   # Drive 上の保存先
os.makedirs(save_dir, exist_ok=True)
dst = shutil.copy(video_path, save_dir)               # コピー先のフルパス
print(f"Drive に保存しました: {dst}")

## 完了

Genesis が Colab 上で動作することを確認できました。
次は `notebooks/02_robogen_colab.ipynb` で RoboGen のパイプライン（LLM タスク生成 → シーン構築 → スキル学習）に進んでください。

うまく動かない場合は `docs/troubleshooting.md` を参照するか、セル5（セルフチェック）の出力を添えて報告してください。